# GPTQ gs64 + FOEM: Qwen3.8-27B (caminho PROVADO do vault)

Pipeline calibrado de quantização INT4 (usa a **config universal que BATE BF16**: gs64 + FOEM + skip-list v8).

Model: `Qwen/Qwen3.8-27B` — Qwen3.5 ForConditionalGeneration, denso 27B (48 GDN + 16 full attn), multimodal.

| Passo | O quê | Detalhe |
|---|---|---|
| Config | `bits=4, group_size=64, sym, desc_act, FOEM α=0.25 β=0.2` | universal (vault) |
| Skip v8 | `linear_attn`, `visual`, `mtp`, `lm_head`, `embed_tokens`, `norm.*` | canonical llmcompressor recipe (Arien0) |
| Calib | `neuralmagic/LLM_compression_calibration` 512 amostras (genérico) | não precisa calib de código |
| Saída | GPTQ pack-quantized → **Marlin nativo vLLM** (~14 GB) | vs BF16 55 GB |
| Eval | HumanEval pass@1 via lm-eval/vLLM **antes** de publicar | regra do vault |


> GPTQ NÃO usa rotação Hadamard (teorema + `feedback_hadamard_doesnt_help_gptq`): GPTQ já modela a distribuição dos pesos.
> Estágios resumíveis + verificação a cada fronteira (regra do Caio: célula-por-célula).


In [ ]:
# ########## Instalação (rode UMA vez; depois RESTART se necessário) ##########
!pip install -q --force-reinstall numpy==2.2.6 scipy  # gptqmodel exige numpy 2.2.6 (ABI)
!pip install -q gptqmodel  # GPTQModel v6+ (ModelCloud)
!pip install -q datasets accelerate safetensors huggingface_hub sentencepiece tiktoken


import numpy, scipy
print('numpy', numpy.__version__, '| scipy', scipy.__version__)
print('='*50)
print('*** Se o kernel pedir RESTART, faça (Runtime -> Restart session) e pule esta célula. ***')


In [ ]:
import torch, transformers
import gptqmodel
print('torch       ', torch.__version__)
print('transformers', transformers.__version__)
print('gptqmodel   ', gptqmodel.__version__)
import psutil; print('CPU RAM', psutil.virtual_memory().total/1e9, 'GB')
if torch.cuda.is_available():
    p=torch.cuda.get_device_properties(0); print('GPU', p.name, p.total_memory/1e9, 'GB')
else: print('NO CUDA')


In [ ]:
import os, json
MODEL = 'Qwen/Qwen3.8-27B'
HF_USER = 'caiovicentino1'
OUT_D = '/content/gptq_qwen38_27b'
HF_REPO = 'caiovicentino1/Qwen3.8-27B-GPTQ-gs64-FOEM'
os.makedirs(OUT_D, exist_ok=True)


# ########## Config universal (vault v7) + skip-list v8 ##########
from gptqmodel import GPTQModel
from gptqmodel.quantization import QuantizeConfig
from gptqmodel.quantization.config import FOEMConfig


quantize_config = QuantizeConfig(
    bits=4,
    group_size=64,            # KEY: 2x mais fino que 128
    sym=True,
    desc_act=True,
    foem=FOEMConfig(alpha=0.25, beta=0.2, device='auto'),
    dynamic={
        '-:.*linear_attn.*': {},   # GDN gates catastróficos se quantizar
        '-:.*visual.*': {},        # tower vision sensível a precisão
        '-:.*mtp.*': {},           # cabeça MTP degrada sob INT4
        '-:lm_head': {},           # projeção de saída
        '-:model.language_model.embed_tokens': {},  # embedding
        '-:.*norm.*': {},          # norms - orçamento pequeno, impacto grande
    },
)
print('QuantizeConfig OK: bits=4 gs=64 sym desc_act FOEM + v8 skip-list')


In [ ]:
import time, torch
# ########## Carregar base + preparar calibração ##########
t0=time.time()
model = GPTQModel.from_pretrained(
    MODEL, quantize_config=quantize_config,
    torch_dtype=torch.bfloat16, trust_remote_code=True, device_map='auto',
    low_cpu_mem_usage=True,
)
print(f'Loaded {MODEL} in {time.time()-t0:.0f}s  (device_map auto: modelo espalha CPU+GPU)')


# Calibração (genérica, 512 amostras - igual ao que BATEU BF16)
from datasets import load_dataset
calib_ds = load_dataset('neuralmagic/LLM_compression_calibration', split='train')
def calib_gen():
    for row in calib_ds:
        yield row['text']
print('Calib dataset pronto:', len(calib_ds), 'amostras de neuralmagic/LLM_compression_calibration')


In [ ]:
import time
# ########## QUANTIZAR (etapa longa - VERIFIQUE antes de seguir) ##########
t0=time.time()
model.quantize(calib_gen())
print(f'Quantize concluído em {(time.time()-t0)/60:.1f} min')
# Estado crítico: se a VM morrer aqui, re-provisionar e refazer (40-30min).


In [ ]:
import time
# ########## Salvar quantizado + FIX tokenizer + validar ##########
t0=time.time()
model.save_quantized(OUT_D)
print(f'save_quantized em {(time.time()-t0)/60:.1f} min')
# vLLM/Marlin + fix do tokenizer-class
import json
tp = f'{OUT_D}/tokenizer_config.json'
with open(tp) as f: tc = json.load(f)
tc['tokenizer_class'] = 'PreTrainedTokenizerFast'
with open(tp, 'w') as f: json.dump(tc, f, indent=2)
print('tokenizer_class fixado -> PreTrainedTokenizerFast')


# checagens
size = sum(os.path.getsize(f'{OUT_D}/{f}') for f in os.listdir(OUT_D) if f.endswith('.safetensors'))/1e9
print(f'Modelo quantizado: {size:.2f} GB (vs ~55 GB BF16)')
import os
assert any(f.endswith('.safetensors') for f in os.listdir(OUT_D)), 'sem safetensors!'
print('OK - artefatos presentes')


In [ ]:
# ########## Upload pro HF ##########
HF_TOKEN=''  # @param {type:'string'}
if HF_TOKEN:
    os.environ['HF_TOKEN']=HF_TOKEN
    from huggingface_hub import HfApi
    api=HfApi()
    api.create_repo(HF_REPO, repo_type='model', exist_ok=True, private=False)
    api.upload_folder(repo_id=HF_REPO, repo_type='model', folder_path=OUT_D)
    print('UPADO:', 'https://huggingface.co/'+HF_REPO)
else:
    print('[!] HF_TOKEN vazio - defina e re-execute')


print('\nDeploy (vLLM, Marlin nativo):')
print(f'  vllm serve {HF_REPO} --gpu-memory-utilization 0.85 --max-model-len 131072')
print('\nEval ANTES de publicar (regra do vault):')
print('  lm_eval --model vllm --tasks humaneval --batch_size 1 ', end='')
print('--model_args pretrained='+HF_REPO+' ,gpu_memory_utilization=0.8,enforce_eager=True,max_model_len=16384')


In [ ]:
# ########## EVAL: ambiente (vLLM + lm-eval) ##########
# vault: vLLM instala 'quebra' transformers -> repinar DEPOIS
!pip install -q vllm lm-eval
!pip install -q --force-reinstall transformers==5.5.0
import torch, vllm, lm_eval
print('vllm', vllm.__version__, '| lm_eval', lm_eval.__version__, '| GPU', torch.cuda.get_device_name(0))


In [ ]:
# ########## EVAL COMPLETO via vLLM (Marlin nativo) - ANTES de publicar ##########
# Rode cada task e anote os scores. Comece por humaneval (a métrica que provou 'bate BF16').
import os, json
HF_TOKEN=''  # @param {type:'string'}
os.environ['HF_TOKEN']=HF_TOKEN
os.environ['VLLM_WORKER_MULTIPROC_METHOD']='spawn'
MODEL_EVAL = HF_REPO      # repo do quantizado (ou OUT_D local)
TASKS = ['humaneval', 'gsm8k', 'mmlu']


# === PPL vs BF16 (wikitext) — rodado no MESMO modelo quantizado ===
import math
from transformers import AutoModelForCausalLM, AutoTokenizer
# nota: para comparar de verdade, mede BF16 original e quantizado nos mesmos chunks


results={}
import lm_eval
for task in TASKS:
    print(f'\n===== {task} =====')
    try:
        res = lm_eval.simple_evaluate(
            model='vllm',
            model_args=f'pretrained={MODEL_EVAL},gpu_memory_utilization=0.8,enforce_eager=True,max_model_len=16384,tensor_parallel_size=1',
            tasks=[task], batch_size=1,
        )
        k = [k for k in res['results'] if res['results'][k]][0]
        scores = {kk: vv for kk, vv in res['results'][k].items() if isinstance(vv, float)}
        results[task] = scores
        print(json.dumps(scores, indent=2))
    except Exception as e:
        print(f'  {task} falhou: {type(e).__name__}: {str(e)[:150]}')


print('\n===== RESUMO EVAL =====')
for t, s in results.items():
    top = [k for k in s if k not in ('alias','num_examples')]
    print(f'  {t:<10}', {k: round(s[k],4) for k in top})
with open('/content/eval_results.json','w') as f: json.dump(results, f, indent=2)
print('\n(use tensor_parallel=1, ou >1 se passar da VRAM)')
